# Music Annotation Project
## Notebook 73: Frozen-SAFE_REMOVE Treatment-Strength Counterfactual Experiment

**Runtime: CPU.** This notebook does not rerun or alter V8.8 classification.

### Scientific question
Among pixels already selected by the frozen V8.8 `SAFE_REMOVE` decision, does moving the output more strongly toward the frozen local paper estimate remove visible residual annotation without changing any additional coordinates or damaging printed notation?

### Fixed boundary
The experiment freezes all selection logic:

- authoritative mask unchanged;
- `p_print` unchanged;
- `p_remove` unchanged;
- `SAFE_REMOVE` unchanged;
- M3 and M5_REMOVE unchanged;
- no new pixel becomes eligible;
- no threshold, model or calibrator changes;
- no clean truth is used;
- every challenger differs from frozen V8.8 only on pixels already in `SAFE_REMOVE`.

### Treatment arms
For a SAFE_REMOVE pixel with frozen V8.8 value `v` and frozen local paper estimate `p`:

- **T0**: frozen V8.8, `v`
- **T50**: `v + 0.50 × max(p-v, 0)`
- **T75**: `v + 0.75 × max(p-v, 0)`
- **T100**: exact local paper estimate, `p`

The one-sided `max(p-v, 0)` rule prevents a challenger from darkening a pixel. T100 is the direct test of whether current residual annotation is caused by under-treatment after a correct SAFE_REMOVE decision.

### Interpretation
- If T100 materially cleans pages without new visible print loss or white-hole artefacts, treatment strength is the leading issue.
- If T100 does not affect the visible residue, those pixels were not in SAFE_REMOVE and the issue is selection coverage.
- If T100 cleans residue but creates visible tonal holes or print loss, SAFE_REMOVE includes mixed pixels or direct paper replacement is too crude.

This is a post-lock counterfactual development audit. It has no production-promotion authority.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=4.44,<6" opencv-python-headless pandas numpy pyarrow matplotlib pillow

## 1. Configuration and immutable output directory

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, hmac, json, math, platform, secrets, shutil, sys, zipfile
import cv2, gradio as gr, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NOTEBOOK72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
SOURCE_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'notebook73_safe_remove_treatment_strength'
REVIEWER_CODE='DB'
RANDOM_SEED=730088
REVIEW_PAGES_PER_GROUP=8
PUBLIC_LINK=True

ARMS={'T0':0.00,'T50':0.50,'T75':0.75,'T100':1.00}
REVIEW_ARMS=['T0','T75','T100']
REQUIRED=['prediction_lock.json','notebook72_verdict.json','inputs/runtime_roster.csv','tables/page_output_hashes.csv','pages']

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

SOURCE=Path(SOURCE_RUN_OVERRIDE) if SOURCE_RUN_OVERRIDE else latest_complete(NOTEBOOK72_ROOT,REQUIRED)
if SOURCE is None: raise RuntimeError('No complete Notebook 72 run found.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook73_safe_remove_treatment'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; PAGES=OUT/'pages'; VISUAL=OUT/'visual_audit'; REVIEW=OUT/'review'; SEALED=OUT/'sealed'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,PAGES,VISUAL,REVIEW,SEALED,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Source:',SOURCE)
print('Output:',OUT)
print('Gradio:',gr.__version__)

## 2. Integrity helpers and source prediction-lock verification

In [ ]:
def require(ok,msg):
    if not bool(ok): raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp'); t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str)); t.replace(path)
def read_gray(path):
    x=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(x is not None,f'Unreadable image: {path}'); return x

def percentile(x,q): return float(np.percentile(x,q)) if len(x) else np.nan

lock=json.loads((SOURCE/'prediction_lock.json').read_text()); verdict=json.loads((SOURCE/'notebook72_verdict.json').read_text())
require(lock.get('state')=='CLOSED','Notebook 72 prediction lock is not closed.')
require(verdict.get('first_failed_gate') is None,'Notebook 72 has a failed gate.')
roster=pd.read_csv(SOURCE/'inputs/runtime_roster.csv',dtype=str).fillna('')
hashes=pd.read_csv(SOURCE/'tables/page_output_hashes.csv',dtype=str).fillna('').set_index('page_key')
require(lock['payload']['runtime_roster_sha256']==sha256_file(SOURCE/'inputs/runtime_roster.csv'),'Notebook 72 roster changed after lock.')
require(lock['payload']['page_output_hashes_sha256']==sha256_file(SOURCE/'tables/page_output_hashes.csv'),'Notebook 72 output hashes changed after lock.')
source_identity={'created_utc':datetime.now(timezone.utc).isoformat(),'source_run':str(SOURCE),'prediction_lock_id':lock['lock_id'],'operator_bundle_id':lock['payload']['operator_bundle_id'],'source_roster_sha256':sha256_file(SOURCE/'inputs/runtime_roster.csv'),'source_output_hashes_sha256':sha256_file(SOURCE/'tables/page_output_hashes.csv'),'arms':ARMS,'review_arms':REVIEW_ARMS,'promotion_authority':False}
atomic_json(INPUTS/'source_identity.json',source_identity)
print(json.dumps(source_identity,indent=2))

## 3. Reconstruct frozen paper values and diagnose treatment opportunity

In [ ]:
page_rows=[]; pixel_rows=[]; failures=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}'; src=SOURCE/'pages'/key
    try:
        observed=read_gray(src/'observed.png'); frozen=read_gray(src/'corrected.png'); safe=read_gray(src/'safe_remove.png')>0; prepared=read_gray(src/'prepared_mask.png')>0
        features=pd.read_parquet(src/'observable_features.parquet')
        require(observed.shape==frozen.shape==safe.shape==prepared.shape,f'{key}: shape mismatch.')
        h=hashes.loc[key]
        require(array_hash(observed)==h.observed_array_sha256,f'{key}: observed hash mismatch.')
        require(array_hash(frozen)==h.corrected_array_sha256,f'{key}: frozen corrected hash mismatch.')
        require(array_hash(safe)==h.safe_remove_array_sha256,f'{key}: SAFE_REMOVE hash mismatch.')
        require({'y','x','paper_value','p_remove','safe_remove'}.issubset(features.columns),f'{key}: feature table lacks required fields.')
        paper=np.full(observed.shape,np.nan,np.float32)
        yy=pd.to_numeric(features.y,errors='raise').astype(int).to_numpy(); xx=pd.to_numeric(features.x,errors='raise').astype(int).to_numpy(); pv=pd.to_numeric(features.paper_value,errors='raise').to_numpy(np.float32)
        paper[yy,xx]=pv
        require(np.isfinite(paper[safe]).all(),f'{key}: SAFE_REMOVE pixels lack paper values.')
        residual=np.maximum(paper.astype(np.float32)-frozen.astype(np.float32),0)
        active=safe&(residual>0)
        vals=residual[safe]
        page_rows.append({'page_key':key,'page':r.page,'population':r.population,'safe_remove_pixels':int(safe.sum()),'safe_pixels_below_paper':int(active.sum()),'fraction_safe_below_paper':float(active.sum()/max(safe.sum(),1)),'mean_distance_to_paper':float(vals.mean()) if len(vals) else 0.0,'median_distance_to_paper':percentile(vals,50),'p75_distance_to_paper':percentile(vals,75),'p90_distance_to_paper':percentile(vals,90),'p95_distance_to_paper':percentile(vals,95),'pixels_10plus_from_paper':int((vals>=10).sum()),'pixels_20plus_from_paper':int((vals>=20).sum()),'pixels_30plus_from_paper':int((vals>=30).sum()),'opportunity_score':float(vals.sum())})
        ys,xs=np.where(safe)
        for y,x in zip(ys,xs):
            pixel_rows.append({'page_key':key,'page':r.page,'population':r.population,'y':int(y),'x':int(x),'observed_value':int(observed[y,x]),'frozen_value':int(frozen[y,x]),'paper_value':float(paper[y,x]),'distance_to_paper':float(residual[y,x]),'p_remove':float(features.loc[(features.y==y)&(features.x==x),'p_remove'].iloc[0]) if ((features.y==y)&(features.x==x)).any() else np.nan})
    except Exception as e: failures.append({'page_key':key,'error':repr(e)})
if failures:
    pd.DataFrame(failures).to_csv(AUDIT/'input_failures.csv',index=False); raise RuntimeError(f'Input failures on {len(failures)} pages.')
page_df=pd.DataFrame(page_rows); pixel_df=pd.DataFrame(pixel_rows)
page_df.to_csv(TABLES/'treatment_opportunity_by_page.csv',index=False); pixel_df.to_parquet(TABLES/'safe_remove_pixel_tonal_audit.parquet',index=False)
print(page_df.groupby('population').agg(pages=('page_key','size'),safe_pixels=('safe_remove_pixels','sum'),below_paper=('safe_pixels_below_paper','sum'),mean_page_distance=('mean_distance_to_paper','mean'),opportunity=('opportunity_score','sum')))

## 4. Build all fixed-support counterfactual pages

In [ ]:
integrity=[]; arm_metrics=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}'; src=SOURCE/'pages'/key; dst=PAGES/key; dst.mkdir(parents=True,exist_ok=False)
    observed=read_gray(src/'observed.png'); frozen=read_gray(src/'corrected.png'); safe=read_gray(src/'safe_remove.png')>0; prepared=read_gray(src/'prepared_mask.png')>0; features=pd.read_parquet(src/'observable_features.parquet')
    paper=np.full(observed.shape,np.nan,np.float32); yy=features.y.astype(int).to_numpy(); xx=features.x.astype(int).to_numpy(); paper[yy,xx]=features.paper_value.to_numpy(np.float32)
    cv2.imwrite(str(dst/'observed.png'),observed); cv2.imwrite(str(dst/'safe_remove.png'),safe.astype(np.uint8)*255)
    for arm,alpha in ARMS.items():
        out=frozen.copy()
        if alpha>0:
            current=frozen[safe].astype(np.float32); target=paper[safe]; values=current+alpha*np.maximum(target-current,0); out[safe]=np.clip(np.rint(values),0,255).astype(np.uint8)
        changed=out!=frozen
        outside_safe=int(np.sum(changed&~safe)); outside_auth=int(np.sum(out[~prepared]!=observed[~prepared])); darkened=int(np.sum(out.astype(np.int16)<frozen.astype(np.int16)))
        require(outside_safe==0,f'{key} {arm}: changed outside SAFE_REMOVE.')
        require(outside_auth==0,f'{key} {arm}: outside-authority drift.')
        require(darkened==0,f'{key} {arm}: treatment darkened pixels.')
        cv2.imwrite(str(dst/f'{arm}.png'),out); cv2.imwrite(str(dst/f'{arm}_difference.png'),changed.astype(np.uint8)*255)
        delta=out.astype(np.int16)-frozen.astype(np.int16)
        arm_metrics.append({'page_key':key,'page':r.page,'population':r.population,'arm':arm,'alpha':alpha,'changed_pixels':int(changed.sum()),'mean_positive_change':float(delta[changed].mean()) if changed.any() else 0.0,'p95_positive_change':percentile(delta[changed],95),'maximum_change':int(delta.max()),'outside_safe_remove_changes':outside_safe,'outside_authority_drift':outside_auth,'darkened_pixels':darkened,'output_array_sha256':array_hash(out)})
    integrity.append({'page_key':key,'safe_remove_array_sha256':array_hash(safe),'prepared_mask_array_sha256':array_hash(prepared),'frozen_array_sha256':array_hash(frozen),'paper_on_safe_sha256':array_hash(paper[safe]),'all_arms_support_subset_safe_remove':True,'outside_authority_drift_zero':True,'no_arm_darkens_pixels':True})
pd.DataFrame(arm_metrics).to_csv(TABLES/'counterfactual_arm_metrics.csv',index=False); pd.DataFrame(integrity).to_csv(AUDIT/'counterfactual_integrity.csv',index=False)
print('Generated',len(roster)*len(ARMS),'counterfactual page outputs.')

## 5. Select a source-balanced diagnosis review queue

In [ ]:
# Per source group: top opportunity, middle opportunity and deterministic random coverage.
rng=np.random.default_rng(RANDOM_SEED); selected=[]; selection_rows=[]
for group,g in page_df.groupby('population'):
    g=g.sort_values('opportunity_score',ascending=False).reset_index(drop=True); n=min(REVIEW_PAGES_PER_GROUP,len(g)); top_n=max(2,n//2); top=g.head(top_n); remaining=g.iloc[top_n:]
    middle_n=min(max(1,n//4),len(remaining)); middle=remaining.iloc[np.linspace(0,max(len(remaining)-1,0),middle_n,dtype=int)] if middle_n else remaining.head(0)
    used=set(top.page_key)|set(middle.page_key); pool=g[~g.page_key.isin(used)]; random_n=n-len(used); random=pool.iloc[rng.choice(len(pool),random_n,replace=False)] if random_n>0 else pool.head(0)
    q=pd.concat([top.assign(selection_reason='top_opportunity'),middle.assign(selection_reason='distribution_middle'),random.assign(selection_reason='deterministic_random')]).drop_duplicates('page_key').head(n)
    selected.extend(q.page_key.tolist()); selection_rows.append(q)
review_roster=pd.concat(selection_rows,ignore_index=True).sort_values(['population','selection_reason','opportunity_score'],ascending=[True,True,False]).reset_index(drop=True); review_roster['review_order']=np.arange(1,len(review_roster)+1)
review_roster.to_csv(REVIEW/'review_roster.csv',index=False)
print(review_roster.groupby(['population','selection_reason']).size())

## 6. Create concealed T0/T75/T100 assignments

In [ ]:
secret=secrets.token_bytes(32); rng=np.random.default_rng(RANDOM_SEED+1); assignments=[]
for _,r in review_roster.iterrows():
    perm=list(rng.permutation(REVIEW_ARMS)); row={'review_order':int(r.review_order),'page_key':str(r.page_key),'population':str(r.population),'A':str(perm[0]),'B':str(perm[1]),'C':str(perm[2])}; row['commitment']=hmac.new(secret,json.dumps(row,sort_keys=True).encode(),hashlib.sha256).hexdigest(); assignments.append(row)
assignment=pd.DataFrame(assignments); assignment.to_csv(SEALED/'review_key.csv',index=False)
atomic_json(SEALED/'review_key_seal.json',{'created_utc':datetime.now(timezone.utc).isoformat(),'key_sha256':sha256_file(SEALED/'review_key.csv'),'secret_hex':secret.hex(),'status':'SEALED_UNTIL_REVIEW_COMPLETE'})
assignment[['review_order','page_key','commitment']].to_csv(REVIEW/'review_queue_public.csv',index=False)
print('Review pages:',len(assignment))

## 7. Optional focused contact sheets before the concealed review

In [ ]:
def make_sheet(rows,name,max_pages=12):
    rows=rows.head(max_pages)
    if rows.empty:return
    fig,axes=plt.subplots(len(rows),4,figsize=(16,max(5,5.5*len(rows))),squeeze=False)
    for i,(_,r) in enumerate(rows.iterrows()):
        key=r.page_key
        for j,(arm,title) in enumerate([('T0','Frozen V8.8'),('T50','50% to paper'),('T75','75% to paper'),('T100','Exact paper')]):
            im=read_gray(PAGES/key/f'{arm}.png'); axes[i,j].imshow(im,cmap='gray',vmin=0,vmax=255); axes[i,j].set_title(f'{key}\n{title}'); axes[i,j].axis('off')
    plt.tight_layout(); plt.savefig(VISUAL/f'{name}.png',dpi=130,bbox_inches='tight'); plt.close(fig)
make_sheet(page_df.sort_values('opportunity_score',ascending=False),'largest_tonal_opportunity_pages')
make_sheet(review_roster.sort_values('review_order'),'review_queue_preview')
print('Contact sheets:',VISUAL)

## 8. Persistent reviewer: judge treatment outcome, not algorithm identity

In [ ]:
RESPONSES=REVIEW/'responses.csv'; HISTORY=REVIEW/'response_history.jsonl'; REVIEW_LOCK=OUT/'review_lock.json'
FIELDS=['review_order','page_key','reviewer_code','submitted_utc','A_playable','A_print_damage','A_annotation_removal','A_tonal_artifact','B_playable','B_print_damage','B_annotation_removal','B_tonal_artifact','C_playable','C_print_damage','C_annotation_removal','C_tonal_artifact','preference_order','diagnosis','confidence','notes','commitment','response_sha256']
if not RESPONSES.exists(): pd.DataFrame(columns=FIELDS).to_csv(RESPONSES,index=False)
def response_frame():
    q=pd.read_csv(RESPONSES,dtype=str,keep_default_na=False)
    for c in FIELDS:
        if c not in q:q[c]=''
    q=q[FIELDS]; q['review_order']=pd.to_numeric(q.review_order,errors='coerce').astype('Int64'); return q
def rhash(row): return object_hash({k:row.get(k,'') for k in FIELDS if k!='response_sha256'})
def done(): return set(response_frame().page_key)
def pending():
    q=assignment[~assignment.page_key.isin(done())].copy(); q['review_order']=pd.to_numeric(q.review_order,errors='raise').astype('Int64'); return q.sort_values('review_order').page_key.tolist()
def save_row(row):
    if REVIEW_LOCK.exists(): raise RuntimeError('Review locked.')
    row={k:row.get(k,'') for k in FIELDS}; row['review_order']=int(row['review_order']); row['response_sha256']=rhash(row); q=response_frame(); old=q[q.page_key==row['page_key']]
    with open(HISTORY,'a') as f:f.write(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'page_key':row['page_key'],'old':None if old.empty else old.iloc[0].astype(str).to_dict(),'new':row},sort_keys=True)+'\n')
    q=q[q.page_key!=row['page_key']]; q=pd.concat([q,pd.DataFrame([row])],ignore_index=True); q['review_order']=pd.to_numeric(q.review_order,errors='raise').astype('Int64'); q.sort_values(['review_order','page_key']).to_csv(RESPONSES,index=False)

def render(key,label): return read_gray(PAGES/key/f'{assignment.set_index("page_key").loc[key,label]}.png')
def defaults(): return ['UNCERTAIN','UNCERTAIN','PARTIAL','UNCERTAIN']*3+['A=B=C','UNCERTAIN','MEDIUM','']
def load_page(key):
    if not key:return [None,None,None,f'Completed {len(done())}/{len(assignment)}']+defaults()
    return [render(key,'A'),render(key,'B'),render(key,'C'),f'Completed {len(done())}/{len(assignment)} | item {int(assignment.set_index("page_key").loc[key,"review_order"])}']+defaults()
def submit(key,*values):
    names=[f'{v}_{f}' for v in ['A','B','C'] for f in ['playable','print_damage','annotation_removal','tonal_artifact']]+['preference_order','diagnosis','confidence','notes']; data=dict(zip(names,values)); required=[k for k in names if k!='notes']
    if not key:raise gr.Error('Select a page.')
    if any(not str(data[k]).strip() for k in required):raise gr.Error('Complete required fields.')
    a=assignment.set_index('page_key').loc[key]; row={'review_order':int(a.review_order),'page_key':key,'reviewer_code':REVIEWER_CODE,'submitted_utc':datetime.now(timezone.utc).isoformat(),'commitment':a.commitment,**data}; save_row(row); q=pending(); return gr.update(choices=q,value=(q[0] if q else None)),f'Saved {key}. Completed {len(done())}/{len(assignment)}'

with gr.Blocks(title='Notebook 73 treatment-strength review') as app:
    gr.Markdown('# Notebook 73 — concealed treatment-strength review\nAll three versions use exactly the same frozen SAFE_REMOVE coordinates. Judge annotation removal, print safety and tonal artefacts independently.')
    with gr.Row(): page=gr.Dropdown(choices=pending(),value=(pending()[0] if pending() else None),label='Review item'); status=gr.Textbox(value=f'Completed {len(done())}/{len(assignment)}',interactive=False)
    with gr.Row(): ia=gr.Image(label='A',height=650); ib=gr.Image(label='B',height=650); ic=gr.Image(label='C',height=650)
    comps=[]
    with gr.Tabs():
        for v in ['A','B','C']:
            with gr.Tab(f'Judge {v}'):
                play=gr.Radio(['YES','YES_WITH_DIFFICULTY','NO','UNCERTAIN'],value='UNCERTAIN',label=f'{v}: playable?')
                damage=gr.Radio(['NONE','MINOR','MATERIAL','DESTROYED','UNCERTAIN'],value='UNCERTAIN',label=f'{v}: printed notation damage')
                removal=gr.Radio(['COMPLETE','MOSTLY_REMOVED','PARTIAL','NONE','UNCERTAIN'],value='PARTIAL',label=f'{v}: annotation removal')
                artifact=gr.Radio(['NONE','MINOR_TONAL_PATCH','OBJECTIONABLE_WHITE_HOLE','OTHER_OBJECTIONABLE','UNCERTAIN'],value='UNCERTAIN',label=f'{v}: treatment artefact')
                comps += [play,damage,removal,artifact]
    pref=gr.Dropdown(['A>B>C','A>C>B','B>A>C','B>C>A','C>A>B','C>B>A','A=B>C','A=C>B','B=C>A','A>B=C','B>A=C','C>A=B','A=B=C'],value='A=B=C',label='Preference order, ties allowed')
    diagnosis=gr.Radio(['STRONGER_TREATMENT_HELPS','CURRENT_TREATMENT_ADEQUATE','STRONGER_TREATMENT_CREATES_ARTIFACTS','VISIBLE_RESIDUE_NOT_AFFECTED','MIXED_OR_UNCERTAIN'],value='UNCERTAIN',label='Page-level diagnosis')
    confidence=gr.Radio(['HIGH','MEDIUM','LOW'],value='MEDIUM',label='Confidence'); notes=gr.Textbox(lines=3,label='Notes'); save=gr.Button('Save and load next',variant='primary')
    outputs=[ia,ib,ic,status]+comps+[pref,diagnosis,confidence,notes]
    page.change(load_page,page,outputs); app.load(load_page,page,outputs); save.click(submit,[page]+comps+[pref,diagnosis,confidence,notes],[page,status]).then(load_page,page,outputs)
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)

## 9. Close review, decode and summarise

In [ ]:
responses=response_frame(); require(len(responses)==len(assignment),f'Review incomplete: {len(responses)}/{len(assignment)}')
for _,r in responses.iterrows(): require(r.response_sha256==rhash(r.to_dict()),f'{r.page_key}: response hash mismatch.')
payload={'prediction_lock_id':lock['lock_id'],'responses_sha256':sha256_file(RESPONSES),'history_sha256':sha256_file(HISTORY),'review_key_sha256':sha256_file(SEALED/'review_key.csv'),'responses':len(responses)}
review_lock={'created_utc':datetime.now(timezone.utc).isoformat(),'state':'CLOSED','review_lock_id':object_hash(payload),'payload':payload}; atomic_json(REVIEW_LOCK,review_lock)
wide=responses.merge(assignment,on=['review_order','page_key','commitment'],validate='one_to_one'); long=[]
for _,r in wide.iterrows():
    for label in ['A','B','C']:
        long.append({'review_order':int(r.review_order),'page_key':r.page_key,'population':r.population,'concealed_label':label,'arm':r[label],'playable':r[f'{label}_playable'],'print_damage':r[f'{label}_print_damage'],'annotation_removal':r[f'{label}_annotation_removal'],'tonal_artifact':r[f'{label}_tonal_artifact'],'preference_order':r.preference_order,'preference_rank':next(i for i,g in enumerate(r.preference_order.split('>'),1) if label in g.split('=')),'diagnosis':r.diagnosis,'confidence':r.confidence,'notes':r.notes})
long=pd.DataFrame(long); long.to_csv(TABLES/'decoded_treatment_reviews.csv',index=False); wide.to_csv(TABLES/'decoded_treatment_reviews_wide.csv',index=False)
summary=[]
for measure in ['playable','print_damage','annotation_removal','tonal_artifact','preference_rank']:
    q=long.groupby(['arm',measure],as_index=False).size().rename(columns={'size':'pages',measure:'level'}); q.insert(0,'measure',measure); summary.append(q)
pd.concat(summary,ignore_index=True).to_csv(TABLES/'review_summary_by_arm.csv',index=False)
wide[['page_key','population','diagnosis','confidence','notes']].to_csv(TABLES/'page_level_diagnoses.csv',index=False)
print(long.groupby(['arm','annotation_removal']).size())
print(long.groupby(['arm','print_damage']).size())

## 10. Experiment verdict and curated package

In [ ]:
review_summary=pd.read_csv(TABLES/'review_summary_by_arm.csv'); decoded=pd.read_csv(TABLES/'decoded_treatment_reviews.csv')
t100=decoded[decoded.arm=='T100']; t0=decoded[decoded.arm=='T0']
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':'NOTEBOOK73_TREATMENT_STRENGTH_EXPERIMENT_COMPLETE','promotion_authority':False,'source_prediction_lock_id':lock['lock_id'],'review_lock_id':review_lock['review_lock_id'],'pages_generated':len(roster),'pages_reviewed':len(assignment),'source_groups_reviewed':int(assignment.population.nunique()),'fixed_support':True,'arms':ARMS,'T100':{'playable_yes':int((t100.playable=='YES').sum()),'material_or_destroyed_print':int(t100.print_damage.isin(['MATERIAL','DESTROYED']).sum()),'complete_or_mostly_removed':int(t100.annotation_removal.isin(['COMPLETE','MOSTLY_REMOVED']).sum()),'objectionable_artifact':int(t100.tonal_artifact.isin(['OBJECTIONABLE_WHITE_HOLE','OTHER_OBJECTIONABLE']).sum())},'T0':{'playable_yes':int((t0.playable=='YES').sum()),'material_or_destroyed_print':int(t0.print_damage.isin(['MATERIAL','DESTROYED']).sum()),'complete_or_mostly_removed':int(t0.annotation_removal.isin(['COMPLETE','MOSTLY_REMOVED']).sum())},'diagnoses':wide.diagnosis.value_counts().to_dict(),'interpretation_rule':{'treatment_supported':'T100/T75 improve removal without material print damage or objectionable tonal artefact.','selection_problem':'Visible residue is unaffected by stronger treatment.','reconstruction_problem':'Stronger treatment improves removal but causes objectionable tonal artefacts.','mixed_support':'Outcomes differ materially by source/page; retain fixed-support evidence and design a scoped challenger.'}}
atomic_json(OUT/'notebook73_verdict.json',record)
lines=['NOTEBOOK 73 — SAFE_REMOVE TREATMENT-STRENGTH COUNTERFACTUAL','',f'Pages generated: {len(roster)}',f'Pages reviewed: {len(assignment)}',f'Source groups reviewed: {assignment.population.nunique()}',f'Prediction lock: {lock["lock_id"]}',f'Review lock: {review_lock["review_lock_id"]}','','All challengers changed only frozen SAFE_REMOVE pixels.','No model, threshold, probability, SAFE_REMOVE decision, M3 or M5_REMOVE coordinate was altered.','',f'T100 playable YES: {record["T100"]["playable_yes"]}',f'T100 material/destroyed print: {record["T100"]["material_or_destroyed_print"]}',f'T100 complete/mostly removed: {record["T100"]["complete_or_mostly_removed"]}',f'T100 objectionable tonal artefact: {record["T100"]["objectionable_artifact"]}','','No production promotion authority.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook73_verdict.json',INPUTS/'source_identity.json',TABLES/'treatment_opportunity_by_page.csv',TABLES/'counterfactual_arm_metrics.csv',AUDIT/'counterfactual_integrity.csv',REVIEW/'review_roster.csv',REVIEW/'responses.csv',OUT/'review_lock.json',TABLES/'decoded_treatment_reviews.csv',TABLES/'review_summary_by_arm.csv',TABLES/'page_level_diagnoses.csv']
for p in required:require(p.exists(),f'Missing package item: {p}')
manifest=[]
for src in required:
    dst=UPLOAD/src.name; shutil.copy2(src,dst); manifest.append({'file':dst.name,'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'README.txt').write_text('Notebook 73 fixed-SAFE_REMOVE treatment-strength experiment. Start with executive_summary.txt and notebook73_verdict.json. No production promotion is granted.\n')
with zipfile.ZipFile(UPLOAD/'notebook73_treatment_strength_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip':z.write(p,p.name)
print('\n'.join(lines)); print('Bundle:',UPLOAD/'notebook73_treatment_strength_bundle.zip')

## Decision boundary

This notebook isolates treatment strength from selection coverage. A positive result supports changing what happens **after** frozen SAFE_REMOVE, not widening SAFE_REMOVE itself.

No arm is eligible for operational use until the result is reviewed, the treatment is separately frozen, and an appropriate regression/validation decision is made.